# DS-01 — Sport and Recreational Facilities List

Profiling the landed object, before the Stage 3 column contract is written.

DS-01 is the spine of the product. Every venue page, every access chain and every
match rate figure is keyed on a row from this file, so its grain, its extent and its
truncation behaviour all have to be established before anything else is built on it.

What this notebook has to settle:

1. **Grain.** Is one row one facility, or one facility per sport played?
2. **Extent.** The publisher is Victoria-wide. How much of it is outside our 31 LGAs?
3. **Coordinates.** How many rows cannot be placed, and where do they go?
4. **Attribute reliability.** `Access_` and `Facility Features` carry the accessible
   toilet and accessible parking tokens. Are those fields complete, or truncated?
5. **LGA naming.** Do publisher names reconcile to the ABS names DS-06 uses?

Nothing here fills a gap. Where the source does not say, the profile records that it
does not say.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import pandas as pd
import profile_lib as pl

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

print("project root:", pl.PROJECT_ROOT)
print("raw zone:    ", pl.RAW_ROOT)


project root: C:\Users\nitin\Documents\Projects\Final_Project\SportAble
raw zone:     C:\Users\nitin\Documents\Projects\Final_Project\SportAble\_raw


In [2]:
raw = pl.resolve("DS-01")
raw

RawObject(DS-01 dt=2026-08-31 srv_ifmd_all-facilities.xlsx 3,656,786B sha=d7a3992688ab… [match])

In [3]:
p = pl.Profile(raw, "Sport and Recreational Facilities List")

p.check(
    "raw_integrity",
    "pass" if raw.sha_matches_manifest else ("info" if raw.sha_matches_manifest is None else "fail"),
    f"SHA-256 of the profiled object is {raw.sha256}",
    raw.sha256,
)
print(raw.sha256)

d7a3992688ab6f007c736ac0ee89928dc5b0fd7b075d1d9dcda49a854a42dc34


## Workbook inventory

The publisher ships several sheets. Only one is the data; the others are derived views and must not be loaded.

In [4]:
xl = pd.ExcelFile(raw.path)
sheets = {}
for s in xl.sheet_names:
    head = pd.read_excel(xl, sheet_name=s, nrows=0)
    n = len(pd.read_excel(xl, sheet_name=s, usecols=[0]))
    sheets[s] = {"rows": int(n), "columns": int(len(head.columns))}
p.observe("sheets", sheets, "publisher ships derived views alongside the data sheet")
pd.DataFrame(sheets).T

,rows,columns
wholeIFMD,9590,31
Alphabetised by facilityname,9790,0
count if 6,9593,0
Sheet1,96,2


In [5]:
SHEET = "wholeIFMD"
df = pd.read_excel(xl, sheet_name=SHEET)
p.observe("sheet_loaded", SHEET, "the other sheets are derived views and are not loaded")
p.contract(f"Load sheet '{SHEET}' only. Reject the file if that sheet name is absent.")
print(f"{len(df):,} rows x {len(df.columns)} columns")
list(df.columns)

9,590 rows x 31 columns


['Facility ID',
 'LGA Name',
 'Facility Name',
 'Facility_AutoNumber',
 'Street #',
 'Street Name',
 'Street Type',
 'Suburb/Town',
 'Pcode',
 'Melway Ref',
 'VicRoads Ref',
 'Latitude',
 'Longitude',
 'Facility Ownership',
 'Facility Purpose',
 'Facility Category',
 'CFA Safer Place?',
 'Access_',
 'FaciltySportPlayedID',
 'Facility ID.1',
 'Sports Played',
 'Number of Field/Courts',
 'Field/Surface Type',
 'Age of Facility',
 'Condition of Facility',
 'Facility Upgrade Age',
 'Changerooms',
 'Facility Features',
 'Spectator numbers for seating/shelter',
 'FullAddress',
 'MelwaysVicRoadsRef']

## 1. Grain

A row count is not a venue count until this is settled.

In [6]:
rows = len(df)
uniq_fac = int(df["Facility ID"].nunique())
p.observe("row_count", rows)
p.observe("unique_facility_id", uniq_fac)
p.observe("rows_per_facility_max", int(df["Facility ID"].value_counts().max()))

is_facility_grain = rows == uniq_fac
p.check(
    "grain",
    "pass" if is_facility_grain else "warn",
    (f"one row per facility ({rows:,})" if is_facility_grain else
     f"{rows:,} rows resolve to {uniq_fac:,} facilities — the sheet is at facility x sport grain, "
     f"deduplication to facility grain is mandatory before any venue count is reported"),
    {"rows": rows, "facilities": uniq_fac},
)
if not is_facility_grain:
    p.contract("Deduplicate on Facility ID before load. Sports Played becomes a child table or a collapsed array, never a duplicated venue row.")

df["Facility ID"].value_counts().head(10)

Facility ID
CASEYC12260    11
MELBOU11026    11
SOUTHG12410    11
WARRNA11676    11
KNOXCI8606     10
GREATE12111    10
WELLIN11034    10
ALPINE10930     9
CENTRA12561     9
GOLDEN8317      9
Name: count, dtype: int64

In [7]:
# Does anything vary within a Facility ID other than the sport columns?
sport_cols = {"FaciltySportPlayedID", "Facility ID.1", "Sports Played",
              "Number of Field/Courts", "Field/Surface Type"}
venue_cols = [c for c in df.columns if c not in sport_cols]
varying = {}
g = df.groupby("Facility ID", dropna=False)
for c in venue_cols:
    nun = g[c].nunique(dropna=False)
    n_bad = int((nun > 1).sum())
    if n_bad:
        varying[c] = n_bad
p.observe("columns_varying_within_facility", varying,
          "columns that disagree across rows of the same facility; each needs a documented collapse rule")
if varying:
    p.check("dedup_collapse_rules", "warn",
            f"{len(varying)} venue-level columns take more than one value within a single Facility ID — "
            f"a first / max / mode rule must be written for each before dedup",
            varying)
    p.contract("Write an explicit collapse rule per column listed in columns_varying_within_facility. Do not rely on drop_duplicates ordering.")
else:
    p.check("dedup_collapse_rules", "pass", "venue-level columns are constant within a Facility ID", {})
varying

{'Age of Facility': 611,
 'Condition of Facility': 701,
 'Facility Upgrade Age': 478,
 'Changerooms': 659,
 'Facility Features': 846,
 'Spectator numbers for seating/shelter': 352}

## 2. Extent

The publisher covers Victoria. The product covers the 31 councils of metropolitan
Melbourne. The authoritative clip is spatial against DS-06; the name reconciliation
below exists to surface naming drift, not to perform the clip.

In [8]:
lga_counts = df["LGA Name"].value_counts(dropna=False)
p.observe("distinct_lga_publisher", int(df["LGA Name"].nunique()))
p.observe("rows_null_lga", int(df["LGA Name"].isna().sum()))
print(f"{df['LGA Name'].nunique()} distinct LGA names in the publisher file")
lga_counts.head(20)

90 distinct LGA names in the publisher file


LGA Name
Geelong City Council                  359
Casey City Council                    243
Bendigo City Council                  240
Knox City Council                     223
Brimbank City Council                 204
Mornington Peninsula Shire Council    201
Yarra Ranges Shire Council            200
Ballarat City Council                 199
Boroondara City Council               183
Shepparton City Council               175
Hume City Council                     174
East Gippsland Shire Council          170
Kingston City Council                 166
Baw Baw Shire Council                 160
Monash City Council                   160
Bayside City Council                  158
Wellington Shire Council              157
Banyule City Council                  156
Cardinia Shire Council                153
LaTrobe City Council                  153
Name: count, dtype: int64

In [9]:
df["lga_norm"] = df["LGA Name"].map(pl.normalise_lga)
gm = set(pl.GREATER_MELBOURNE_LGAS)
present = set(df["lga_norm"].dropna())

matched = sorted(gm & present)
missing = sorted(gm - present)
outside = sorted(present - gm)

in_scope_rows = int(df["lga_norm"].isin(gm).sum())
p.observe("gm_lgas_matched", matched)
p.observe("gm_lgas_missing_from_source", missing)
p.observe("lgas_outside_greater_melbourne", outside)
p.observe("rows_in_greater_melbourne_by_name", in_scope_rows)
p.observe("rows_outside_greater_melbourne_by_name", int(rows - in_scope_rows))

p.check(
    "gm_lga_coverage",
    "pass" if not missing else "warn",
    (f"all 31 Greater Melbourne LGAs appear in the source"
     if not missing else
     f"{len(missing)} of the 31 Greater Melbourne LGAs have no row after name normalisation: {missing}"),
    {"matched": len(matched), "missing": missing},
)
p.check(
    "extent",
    "info",
    f"{in_scope_rows:,} of {rows:,} rows fall in the 31 Greater Melbourne LGAs by name; "
    f"{rows - in_scope_rows:,} rows are elsewhere in Victoria and are out of scope",
    in_scope_rows,
)
p.contract(f"Clip to Greater Melbourne spatially against {pl.LGA_BOUNDARY_ID} before load. Do not filter on LGA name in the API.")

print("matched:", len(matched))
print("missing from source:", missing)
print("outside Greater Melbourne:", len(outside))

matched: 31
missing from source: []
outside Greater Melbourne: 50


In [10]:
# Which raw publisher names required an alias to reconcile?
aliased = (df.loc[df["LGA Name"].notna(), ["LGA Name", "lga_norm"]]
             .drop_duplicates()
             .assign(changed=lambda d: d["LGA Name"].str.strip() != d["lga_norm"]))
renamed = aliased[aliased["lga_norm"].isin(gm) &
                  ~aliased["LGA Name"].str.contains("|".join(gm), case=False, na=False, regex=True)]
p.observe("lga_names_requiring_alias", renamed.to_dict("records"),
          "publisher names that only reconcile to an ABS name through a recorded rename")
if len(renamed):
    p.limitation(
        "DS-01 uses superseded council names for "
        + ", ".join(sorted(renamed["lga_norm"])) + ". The rename is applied from a recorded alias "
        "table, not inferred by fuzzy matching."
    )
renamed

,LGA Name,lga_norm,changed
2509,Dandenong City Council,Greater Dandenong,True
6534,Moreland City Council,Merri-bek,True


## 3. Coordinates

In [11]:
coord_stats = pl.check_coordinates(p, df, "Latitude", "Longitude", label="rows")
null_coord_facilities = int(df.loc[df["Latitude"].isna(), "Facility ID"].nunique())
p.observe("facilities_with_null_coords", null_coord_facilities)
p.contract("Rows with a null or out-of-range coordinate are quarantined with reason code COORD_MISSING. They are never geocoded.")
coord_stats

{'total': 9590,
 'null_coords': 354,
 'out_of_range': 7,
 'likely_swapped_lat_lon': 5,
 'inside_gm_bbox': 5294,
 'outside_gm_bbox': 3942,
 'pct_inside_gm_bbox': 55.2}

## 4. Attribute reliability

`Access_` and `Facility Features` are comma-delimited attribute lists. Between them
they carry the two tokens that matter most to the access chain: an accessible toilet
and an accessible parking bay. If either field is cut at a fixed width, then a token
missing from a row could mean the facility lacks it **or** could mean the list was
truncated before reaching it — and those are not the same answer.

In [12]:
trunc_access = pl.truncation_scan(df["Access_"], "Access_", p)
trunc_feat = pl.truncation_scan(df["Facility Features"], "Facility Features", p)
pd.DataFrame([trunc_access, trunc_feat])

,column,non_null,max_length,rows_at_max_length,share_at_max_length,truncated
0,Access_,3468,116,623,0.1796,False
1,Facility Features,5971,255,1052,0.1762,True


In [13]:
access_tokens = pl.token_counts(df["Access_"])
p.observe("access_vocabulary", access_tokens.to_dict("records"))
access_tokens

,token,rows
0,Sports Pavilion / Clubrooms,2649
1,Parking,2588
2,Changerooms,2580
3,Canteen / Kiosk,2412
4,Spectator Areas,2370
5,Playing Areas,2231
6,Unisex Toilet(s),1411


In [14]:
feat_tokens = pl.token_counts(df["Facility Features"])
frags = pl.fragment_report(feat_tokens)
p.observe("facility_features_vocabulary_size", int(len(feat_tokens)))
p.observe("facility_features_fragments", frags.to_dict("records"),
          "tokens that are prefixes of a longer token — tails of truncated lists, not real values")
p.check(
    "facility_features_fragments",
    "warn" if len(frags) else "pass",
    (f"{len(frags)} of {len(feat_tokens)} tokens in Facility Features are prefixes of longer tokens, "
     f"confirming truncation — these must not be loaded as categories"
     if len(frags) else "no truncation fragments in the Facility Features vocabulary"),
    int(len(frags)),
)
frags.head(30)

,token,rows,likely_truncation_fragment
18,Toilets (Within Sports Pavilion /,53,True
19,Trainers,44,True
20,Trainer,42,True
21,Toilets (Within,42,True
22,Toilets (W,35,True
23,Toilets (Within Sports Pavili,27,True
24,Toilets (Within Sports Pavilion / Clubrooms,27,True
25,Trainers / Medical Faciliti,26,True
26,Toilets (Within Sports Pavilion / Clubroo,25,True
27,Toilets (O,25,True


In [15]:
# The two access-chain tokens, and how many rows we cannot answer for.
ACCESS_TOKENS = {
    "accessible_toilet": "Toilets (Disabled)",
    "accessible_parking": "Parking Bay(s) for the disabled",
}
feat = df["Facility Features"].fillna("").astype(str)
at_max = feat.str.len() >= trunc_feat["max_length"]

tri = {}
for key, token in ACCESS_TOKENS.items():
    present = feat.str.contains(token, regex=False)
    # Absent AND truncated is unknown, not absent.
    unknown = (~present) & at_max
    absent = (~present) & (~at_max)
    tri[key] = {
        "token": token,
        "present": int(present.sum()),
        "absent": int(absent.sum()),
        "unknown_due_to_truncation": int(unknown.sum()),
        "unknown_field_null": int((df["Facility Features"].isna()).sum()),
    }
p.observe("access_chain_tokens_tristate", tri)
p.check(
    "tristate_required",
    "warn",
    "accessible toilet and accessible parking from DS-01 resolve to three states, not two: "
    f"{tri['accessible_toilet']['unknown_due_to_truncation']:,} rows are unknown for accessible toilet "
    "purely because the field was cut",
    tri,
)
p.contract("Derive DS-01 access tokens as tri-state. Absence of a token in a row at maximum field length is NOT_PUBLISHED, never false.")
p.limitation(
    f"DS-01 Facility Features is truncated at {trunc_feat['max_length']} characters. "
    "Presence of an accessible toilet or accessible parking token is evidence; absence is not. "
    "Rows at maximum field length are reported as no published information."
)
pd.DataFrame(tri).T

,token,present,absent,unknown_due_to_truncation,unknown_field_null
accessible_toilet,Toilets (Disabled),2606,6862,122,3619
accessible_parking,Parking Bay(s) for the disabled,2498,6908,184,3619


In [16]:
changerooms = pl.token_counts(df["Changerooms"])
p.observe("changerooms_vocabulary", changerooms.to_dict("records"))
p.observe("rows_null_changerooms", int(df["Changerooms"].isna().sum()))
p.check(
    "changeroom_accessibility",
    "info",
    "the Changerooms vocabulary distinguishes gender and officials, not accessibility — "
    "DS-01 does not publish whether a changeroom is accessible",
    changerooms.to_dict("records"),
)
p.limitation("DS-01 records changeroom presence and gender but never changeroom accessibility. The change link of the access chain is not answerable from this source alone.")
changerooms

,token,rows
0,Male,3819
1,Female,2975
2,Umpire/Officials,1937
3,Unisex,1043
4,Same as above,54


## Save

In [17]:
p.save()

DS-01 — Sport and Recreational Facilities List
  object   srv_ifmd_all-facilities.xlsx  (3,656,786 bytes)
  dt       2026-08-31
  sha256   d7a3992688ab6f007c736ac0ee89928dc5b0fd7b075d1d9dcda49a854a42dc34
  manifest hash matches

  Checks (FAIL overall)
    [PASS] raw_integrity: SHA-256 of the profiled object is d7a3992688ab6f007c736ac0ee89928dc5b0fd7b075d1d9dcda49a854a42dc34
    [WARN] grain: 9,590 rows resolve to 5,000 facilities — the sheet is at facility x sport grain, deduplication to facility grain is mandatory before any venue count is reported
    [WARN] dedup_collapse_rules: 6 venue-level columns take more than one value within a single Facility ID — a first / max / mode rule must be written for each before dedup
    [PASS] gm_lga_coverage: all 31 Greater Melbourne LGAs appear in the source
    [INFO] extent: 4,629 of 9,590 rows fall in the 31 Greater Melbourne LGAs by name; 4,961 rows are elsewhere in Victoria and are out of scope
    [WARN] coords_present: 354 of 9,590 rows h

WindowsPath('C:/Users/nitin/Documents/Projects/Final_Project/SportAble/_profiles/dt=2026-08-31/DS-01.json')